In [1]:
# Télécharge les données JSON des phénotypes et 
# affiche la structure du premier enregistrement pour inspection.
import urllib.request
import json
import pandas as pd
import os

url_phenotypes = "http://www.orphadata.org/cgi-bin/free_product4_hpo.json"

r = urllib.request.urlopen(url_phenotypes)
meta = json.loads(r.read().decode("utf-8"))

# C'est une liste, on affiche le premier élément pour voir la structure
print(f"Type : {type(meta)}")
print(f"Nombre d'éléments : {len(meta)}")
print(f"\nPremier élément :")
print(json.dumps(meta[0], indent=2))

Type : <class 'list'>
Nombre d'éléments : 7

Premier élément :
{
  "anUrl": "http://www.orphadata.org/data/xml/en_product4.xml",
  "aProduct": "en_product4.xml",
  "anAmount": "4258",
  "anId": "1",
  "aType": "xml",
  "aLanguage": "English",
  "aSize": "44.6 MB",
  "aDate": "01 Jul 24"
}


In [2]:
# Télécharge le fichier XML des phénotypes depuis l’URL 
# récupérée et affiche sa taille après téléchargement.
file_url = meta[0]["anUrl"]  # http://www.orphadata.org/data/xml/en_product4.xml
output_file = "orphadata_phenotypes.xml"

print(f"⬇ Téléchargement depuis : {file_url}")
print("⏳ Fichier de 44MB, patiente ~30 secondes...")

urllib.request.urlretrieve(file_url, output_file)
size = os.path.getsize(output_file) // 1024
print(f"✅ Téléchargé : {output_file} ({size} KB)")

⬇ Téléchargement depuis : http://www.orphadata.org/data/xml/en_product4.xml
⏳ Fichier de 44MB, patiente ~30 secondes...
✅ Téléchargé : orphadata_phenotypes.xml (46770 KB)


In [3]:
# Analyse le fichier XML d’Orphadata pour extraire les maladies et leurs symptômes, 
# puis filtre celles présentant des signes liés aux muscles, nerfs, os ou articulations. 
# `ET.parse()` sert à lire le XML, tandis que 
# `DataFrame` et `str.contains()` facilitent l’organisation et le filtrage des données.

import xml.etree.ElementTree as ET

def parse_orphadata_xml(fichier_xml):
    print(f"📖 Parsing {fichier_xml}...")
    tree = ET.parse(fichier_xml)
    root = tree.getroot()
    
    records = []
    disorders = root.findall(".//Disorder")
    print(f"📦 {len(disorders)} maladies trouvées")
    
    for disorder in disorders:
        name_el  = disorder.find("Name")
        code_el  = disorder.find("OrphaCode")
        nom      = name_el.text  if name_el  is not None else ""
        code     = code_el.text  if code_el  is not None else ""
        
        hpo_terms = [
            hpo.find(".//HPOTerm").text
            for hpo in disorder.findall(".//HPODisorderAssociation")
            if hpo.find(".//HPOTerm") is not None
        ]
        
        records.append({
            "maladie":      nom,
            "orphacode":    code,
            "nb_symptomes": len(hpo_terms),
            "symptomes":    " | ".join(hpo_terms[:10]),
            "source":       f"https://www.orpha.net/en/disease/detail/{code}",
            "origine":      "Orphadata"
        })
    
    return records

data = parse_orphadata_xml(output_file)
df_orphanet = pd.DataFrame(data)

# Filtre pertinent pour elaraMed
keywords = ["pain", "muscle", "nerve", "joint", "bone",
            "myop", "neurop", "arthr", "spinal", "skeletal", "weakness"]

mask = df_orphanet["symptomes"].str.lower().str.contains(
    "|".join(keywords), na=False
)
df_filtered = df_orphanet[mask].reset_index(drop=True)

print(f"\n✅ Total maladies Orphanet    : {len(df_orphanet)}")
print(f"✅ Maladies filtrées : {len(df_filtered)}")
df_filtered.head(10)

📖 Parsing orphadata_phenotypes.xml...
📦 4357 maladies trouvées

✅ Total maladies Orphanet    : 4357
✅ Maladies filtrées : 1677


,maladie,orphacode,nb_symptomes,symptomes,source,origine
0,Alpha-mannosidosis,61,43,Macroglossia | Coarse facial features | Hearin...,https://www.orpha.net/en/disease/detail/61,Orphadata
1,Glycogen storage disease due to glycogen debra...,366,7,"Full cheeks | Intellectual disability, mild | ...",https://www.orpha.net/en/disease/detail/366,Orphadata
2,Krabbe disease,487,50,Hearing impairment | Sensorineural hearing imp...,https://www.orpha.net/en/disease/detail/487,Orphadata
3,Glycogen storage disease due to muscle phospho...,371,6,Muscle weakness | Anemia | Hyperuricemia | Myo...,https://www.orpha.net/en/disease/detail/371,Orphadata
4,Sialidosis type 1,812,42,Thick lower lip vermilion | Coarse facial feat...,https://www.orpha.net/en/disease/detail/812,Orphadata
5,Mucolipidosis type III,577,36,Waddling gait | Reduced bone mineral density |...,https://www.orpha.net/en/disease/detail/577,Orphadata
6,Sandhoff disease,796,19,Macrocephaly | Full cheeks | Hearing impairmen...,https://www.orpha.net/en/disease/detail/796,Orphadata
7,Mucopolysaccharidosis type 7,584,26,Inguinal hernia | Coarse facial features | Sho...,https://www.orpha.net/en/disease/detail/584,Orphadata
8,Beta-thalassemia,848,21,Abnormality of the skeletal system | Pallor | ...,https://www.orpha.net/en/disease/detail/848,Orphadata
9,Facioscapulohumeral dystrophy,269,37,Mask-like facies | Skeletal muscle atrophy | E...,https://www.orpha.net/en/disease/detail/269,Orphadata


In [4]:
# Sauvegarde les maladies filtrées dans un fichier CSV pour permettre leur réutilisation et 
# leur analyse ultérieure. `index=False` évite d’ajouter les indices du DataFrame dans le fichier.
df_filtered.to_csv("data_orphanet.csv", index=False)
print(f"💾 data_orphanet.csv sauvegardé — {len(df_filtered)} lignes")

💾 data_orphanet.csv sauvegardé — 1677 lignes


In [5]:
# Interroge l’API de l’OMS pour récupérer les indicateurs liés à des mots-clés médicaux,
# puis les stocke dans un DataFrame en supprimant les doublons. `requests.get()` envoie 
# la requête à l’API, `time.sleep()` limite la fréquence des appels, et `drop_duplicates()` 
# évite les indicateurs répétés.
import requests
import pandas as pd
import time

def fetch_who_indicators(keywords):
    base_url = "https://ghoapi.azureedge.net/api/Indicator"
    records = []
    
    for kw in keywords:
        print(f"🔍 OMS → '{kw}'...")
        try:
            r = requests.get(
                base_url,
                params={"$filter": f"contains(tolower(IndicatorName),'{kw.lower()}')"},
                timeout=15
            )
            if r.status_code != 200:
                print(f"  ⚠ Erreur {r.status_code}")
                continue
            
            items = r.json().get("value", [])
            for item in items:
                records.append({
                    "indicateur": item.get("IndicatorName", ""),
                    "code":       item.get("IndicatorCode", ""),
                    "keyword":    kw,
                    "source":     f"https://www.who.int/data/gho/data/indicators/indicator-details/GHO/{item.get('IndicatorCode','')}",
                    "origine":    "OMS/WHO"
                })
            print(f"  ✔ {len(items)} indicateurs")
            time.sleep(0.5)
        except Exception as e:
            print(f"  ❌ {e}")
    
    return records

keywords_who = ["pain", "musculoskeletal", "neurological", 
                "arthritis", "bone", "disability"]

who_data = fetch_who_indicators(keywords_who)
df_who = pd.DataFrame(who_data).drop_duplicates(subset="code")
print(f"\n✅ Total indicateurs OMS : {len(df_who)}")
df_who.head()

🔍 OMS → 'pain'...
  ✔ 2 indicateurs
🔍 OMS → 'musculoskeletal'...
  ✔ 0 indicateurs
🔍 OMS → 'neurological'...
  ✔ 0 indicateurs
🔍 OMS → 'arthritis'...
  ✔ 0 indicateurs
🔍 OMS → 'bone'...
  ✔ 1 indicateurs
🔍 OMS → 'disability'...
  ✔ 45 indicateurs

✅ Total indicateurs OMS : 48


,indicateur,code,keyword,source,origine
0,Legally-binding controls on lead paint,LEADCONTROL,pain,https://www.who.int/data/gho/data/indicators/i...,OMS/WHO
1,Oral health: Urgent treatment for providing em...,ORALHEALTH_AVAILABILITY_URGENTCARE,pain,https://www.who.int/data/gho/data/indicators/i...,OMS/WHO
2,General availability of bone marrow transplant...,NCD_CCS_bone_marrow,bone,https://www.who.int/data/gho/data/indicators/i...,OMS/WHO
3,Years of healthy life lost to disability (YLDs...,LEPTO_YLD100K,disability,https://www.who.int/data/gho/data/indicators/i...,OMS/WHO
4,Disability-adjusted life years (DALYs) due to ...,LEPTO_DALY,disability,https://www.who.int/data/gho/data/indicators/i...,OMS/WHO


In [6]:
# Récupère les statistiques des indicateurs OMS et les stocke dans un DataFrame pour analyse. 
def fetch_who_stats(codes, max_codes=8):
    records = []
    
    for code in codes[:max_codes]:
        print(f"📊 Stats OMS → {code}")
        try:
            r = requests.get(
                f"https://ghoapi.azureedge.net/api/{code}",
                params={"$top": 30},
                timeout=15
            )
            if r.status_code != 200:
                continue
            
            for item in r.json().get("value", []):
                records.append({
                    "code":  code,
                    "pays":  item.get("SpatialDim", ""),
                    "annee": item.get("TimeDim", ""),
                    "valeur":item.get("NumericValue", ""),
                    "sexe":  item.get("Dim1", ""),
                    "origine": "OMS/WHO Stats"
                })
            time.sleep(0.5)
        except Exception as e:
            print(f"  ❌ {code} : {e}")
    
    return records

top_codes = df_who["code"].tolist()
who_stats = fetch_who_stats(top_codes, max_codes=8)
df_who_stats = pd.DataFrame(who_stats)
print(f"\n✅ Total stats OMS : {len(df_who_stats)} enregistrements")
df_who_stats.head()

📊 Stats OMS → LEADCONTROL
📊 Stats OMS → ORALHEALTH_AVAILABILITY_URGENTCARE
📊 Stats OMS → NCD_CCS_bone_marrow
📊 Stats OMS → LEPTO_YLD100K
📊 Stats OMS → LEPTO_DALY
📊 Stats OMS → LEPTO_DALY100K
📊 Stats OMS → LEPTO_YLD
📊 Stats OMS → RSUD_27

✅ Total stats OMS : 120 enregistrements


,code,pays,annee,valeur,sexe,origine
0,LEADCONTROL,BGD,2018,None,NaN,OMS/WHO Stats
1,LEADCONTROL,LUX,2008,None,NaN,OMS/WHO Stats
2,LEADCONTROL,TKM,2024,None,NaN,OMS/WHO Stats
3,LEADCONTROL,MLT,2008,None,NaN,OMS/WHO Stats
4,LEADCONTROL,ISR,2019,None,NaN,OMS/WHO Stats


In [7]:
# Sauvegarde les données collectées dans des fichiers CSV puis affiche un bilan
# récapitulatif du nombre d’enregistrements récupérés. 
# `to_csv()` permet de conserver les résultats pour une utilisation ultérieure.
df_who.to_csv("data_who_indicateurs.csv", index=False)
df_who_stats.to_csv("data_who_stats.csv", index=False)

print("=" * 45)
print("       📋 BILAN COLLECTE COMPLÈTE")
print("=" * 45)
print(f"  Orphanet — maladies filtrées : {len(df_filtered):>5}")
print(f"  OMS — indicateurs            : {len(df_who):>5}")
print(f"  OMS — données chiffrées      : {len(df_who_stats):>5}")
print("=" * 45)
print(f"  TOTAL                        : {len(df_filtered)+len(df_who)+len(df_who_stats):>5} lignes")
print("\n💾 Fichiers sauvegardés :")
print("  ✔ data_orphanet.csv")
print("  ✔ data_who_indicateurs.csv")
print("  ✔ data_who_stats.csv")

       📋 BILAN COLLECTE COMPLÈTE
  Orphanet — maladies filtrées :  1677
  OMS — indicateurs            :    48
  OMS — données chiffrées      :   120
  TOTAL                        :  1845 lignes

💾 Fichiers sauvegardés :
  ✔ data_orphanet.csv
  ✔ data_who_indicateurs.csv
  ✔ data_who_stats.csv
